# Layer 1: bảng RQ1/RQ2

Mỗi cell bên dưới tính **một model**, in đúng một dòng của bảng trên slide và bảng từng fold của model đó.
Cell cuối ghi bảng hoàn chỉnh vào `output/rq1_rq2_baselines/final/`.

| Bước | Ở đâu |
|---|---|
| Điểm 4 phần mỗi fold: `tb` 560K benign train, `vb`/`va` benign/attack của `val_binary`, `zs` `test_zeroshot` | `layer1_eval/scores.py` (checkpoint → điểm trong `scorers.py`) |
| Ngưỡng: τ = trung bình 14 giá trị Q₁₋α(điểm `tb` của fold), α = 0.5%; alert nếu điểm ≥ τ. Val/test không tham gia đặt ngưỡng | `layer1_eval/metrics.py` |
| F1, FPR trên `val_binary` (benign + 13 loại đã biết); ZS-Acc trên `test_zeroshot` (loại bị giữ lại); trung bình 14 fold | `metrics.fold_metrics` |
| pAUC[0,3%]: diện tích dưới đường ZS-Acc theo FPR thực tế trên benign val, FPR ∈ [0, 3%], chia 3% (ngẫu nhiên = 0.015) | `metrics.pauc` |
| Chọn checkpoint (paper / baseline-study) cho từng dòng | `tables.ROWS` |

Điểm được đọc từ cache (`output/rq1_rq2_layer1/sweep_all_scores.npz`, `output/rq1_rq2_baselines/scores_<model>.npz`).
`RECOMPUTE = True` chấm lại từ checkpoint: vài phút đến vài chục phút mỗi model (model theo fold phải chấm 560K benign ở mỗi fold).
Dữ liệu và checkpoint lấy theo biến môi trường trong `layer1_eval/paths.py` (mặc định repo nằm trong workspace).

In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")   # may 16 GB
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")    # cham tren CPU; phai dat truoc khi import torch
import pandas as pd
from layer1_eval import tables

pd.set_option("display.precision", 4)
RECOMPUTE = False   # True: cham lai tu checkpoint thay vi doc cache
results = {}

## Supervised

In [ ]:
# XGBoost-only: khoa "xgboost_only", checkpoint paper
results["xgb"] = tables.run("xgb", RECOMPUTE)
tables.fold_table(results["xgb"])

In [ ]:
# LightGBM: khoa "lgbm", checkpoint baseline-study
results["lgbm"] = tables.run("lgbm", RECOMPUTE)
tables.fold_table(results["lgbm"])

In [ ]:
# Random Forest: khoa "random_forest", checkpoint baseline-study
results["rf"] = tables.run("rf", RECOMPUTE)
tables.fold_table(results["rf"])

In [ ]:
# KNN: khoa "knn", checkpoint baseline-study
results["knn"] = tables.run("knn", RECOMPUTE)
tables.fold_table(results["knn"])

In [ ]:
# SVM (linear): khoa "svm_only", checkpoint baseline-study
results["svm"] = tables.run("svm", RECOMPUTE)
tables.fold_table(results["svm"])

## One-class

In [ ]:
# AE-only: khoa "ae_only", checkpoint paper
results["ae_only"] = tables.run("ae_only", RECOMPUTE)
tables.fold_table(results["ae_only"])

In [ ]:
# VAE-only: khoa "vae_only", checkpoint baseline-study
results["vae_only"] = tables.run("vae_only", RECOMPUTE)
tables.fold_table(results["vae_only"])

In [ ]:
# Mixture Deep SVDD: khoa "svdd_only", checkpoint baseline-study
results["mix_svdd"] = tables.run("mix_svdd", RECOMPUTE)
tables.fold_table(results["mix_svdd"])

In [ ]:
# Deep SVDD: khoa "deep_svdd", checkpoint baseline-study
results["deep_svdd"] = tables.run("deep_svdd", RECOMPUTE)
tables.fold_table(results["deep_svdd"])

In [ ]:
# OptiForest: khoa "optiforest", checkpoint baseline-study
results["optiforest"] = tables.run("optiforest", RECOMPUTE)
tables.fold_table(results["optiforest"])

## Contrastive

In [ ]:
# CLAD: khoa "clad", checkpoint baseline-study
results["clad"] = tables.run("clad", RECOMPUTE)
tables.fold_table(results["clad"])

## Hybrid

In [ ]:
# VAE-XGBoost: khoa "vae_xgboost", checkpoint baseline-study
results["vae_xgb"] = tables.run("vae_xgb", RECOMPUTE)
tables.fold_table(results["vae_xgb"])

In [ ]:
# AE-XGBoost (ours): khoa "ae_xgboost_mono", checkpoint paper
results["ae_xgb"] = tables.run("ae_xgb", RECOMPUTE)
tables.fold_table(results["ae_xgb"])

In [ ]:
# SVDD-XGBoost (ours): khoa "svdd_xgboost", checkpoint baseline-study
results["svdd_xgb"] = tables.run("svdd_xgb", RECOMPUTE)
tables.fold_table(results["svdd_xgb"])

## Bảng hoàn chỉnh

Ghi `table_main.csv`, `curve.csv`, `thresholds_main.csv`, `final_results.md`, `manifest.json`.
Cột ms/flow đọc từ `output/rq1_rq2_baselines/baseline_latency.json` (đo trong `layer1_extras.ipynb`).

In [ ]:
tables.write_final(results)